# Boltz-2 fork — optimizations, measured and validated

This notebook exercises everything this fork adds on top of stock Boltz-2 2.2.1, and checks it
rather than assuming it:

1. **Correctness without a GPU** — the CPU test suite, which asserts that `--opt_profile exact`
   is *bitwise* `--opt_profile off` through a whole diffusion roll-out, and bounds each `fast`
   lever against the stock path it replaces.
2. **Speed with a GPU** — wall time for `off` / `exact` / `fast` on the same input and seed.
3. **Output agreement** — `exact` must be byte-identical to `off`; `fast` is held to a
   coordinate-RMSD tolerance.
4. **The structure-cache affinity path** — its speedup *and* how far its affinity numbers move
   from the standard two-pass procedure. That path is an alternative inference procedure, so
   this comparison is the point of running it, not a formality.

## Runtime

`Runtime > Change runtime type > A100 GPU` (or L4/V100 — anything of compute capability 8.0+;
a T4 is 7.5 and the SDPA lever is disabled there). Sections 1–3 run on a CPU runtime too.

Weights are about 3 GB. Mount Drive in section 3 if you plan to re-run this notebook.

## 0. What this runtime can do

In [ ]:
import json, os, platform, shutil, subprocess, sys, textwrap

def _smi():
    if not shutil.which("nvidia-smi"):
        return []
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,compute_cap,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False,
    )
    rows = []
    for line in out.stdout.strip().splitlines():
        name, driver, cap, mem = [v.strip() for v in line.split(",")]
        rows.append({"name": name, "driver": driver, "compute_capability": cap,
                     "memory_mib": int(mem)})
    return rows

GPUS = _smi()
HAS_GPU = bool(GPUS)
CAPABILITY = tuple(int(p) for p in GPUS[0]["compute_capability"].split(".")) if HAS_GPU else (0, 0)

# The fork's levers need sm_80 for SDPA; below that Boltz-2 switches it off itself.
FORK_GPU_READY = HAS_GPU and CAPABILITY >= (8, 0)

print(json.dumps({"python": sys.version.split()[0], "platform": platform.platform(),
                  "gpus": GPUS}, indent=2))
print()
print("timed sections:", "yes" if FORK_GPU_READY else
      "no - sections 1-3 only (they need no GPU and no weights)")
if HAS_GPU and not FORK_GPU_READY:
    print(f"  {GPUS[0]['name']} is compute capability {GPUS[0]['compute_capability']};"
          " the SDPA lever needs 8.0+ and Boltz-2 disables it below that.")

## 1. Get the fork

This fork is not on PyPI. Pick how to bring it into the runtime:

* `git`   — clone the remote (the default; nothing to set up).
* `drive` — the repository sits in your Google Drive, which survives runtime restarts and lets
  you test uncommitted changes.
* `local` — you already uploaded or extracted it into the runtime.

In [ ]:
#@title Source of the fork { display-mode: "form" }
SOURCE = "git"  #@param ["git", "drive", "local"]
DRIVE_REPO_PATH = "/content/drive/MyDrive/boltz_truncated"  #@param {type:"string"}
GIT_URL = "https://github.com/aadit-mahajan/boltz_truncated.git"  #@param {type:"string"}
GIT_REF = "main"  #@param {type:"string"}
LOCAL_REPO_PATH = "/content/boltz_truncated"  #@param {type:"string"}

from pathlib import Path

if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    REPO = Path(DRIVE_REPO_PATH)
elif SOURCE == "git":
    assert GIT_URL, "Set GIT_URL, or switch SOURCE to drive/local."
    REPO = Path("/content/boltz_truncated")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(REPO)],
                       check=True)
else:
    REPO = Path(LOCAL_REPO_PATH)

assert (REPO / "boltz" / "opt.py").is_file(), (
    f"{REPO} does not look like this fork (no boltz/opt.py). "
    "Check the path above."
)
print("repository:", REPO)
print("lock:", json.loads((REPO / "optimizations.lock.json").read_text())["upstream"])

## 2. Install and run the CPU test suite

The suite needs no GPU and no weights. It is the part of this notebook that establishes
*correctness*; everything after it measures speed.

Installing from a Drive path can be slow — the editable install writes only a link, but pip
still reads the tree. On a Drive-backed repo, copying to local disk first is much faster.

In [ ]:
WORK = Path("/content/boltz_fork")
if SOURCE == "drive":
    # Editable installs and pytest against a Drive mount are painfully slow.
    if not WORK.exists():
        subprocess.run(
            ["rsync", "-a", "--exclude", ".git", "--exclude", ".venv*",
             "--exclude", "__pycache__", f"{REPO}/", f"{WORK}/"], check=True)
    RUN_REPO = WORK
else:
    RUN_REPO = REPO

print("interpreter:", sys.version.split()[0], "at", sys.executable)


def pip_install(*arguments):
    """Install, and show pip's own message when it refuses.

    A bare CalledProcessError says nothing about *why* a resolution failed --
    an unsupported interpreter and a conflicting pin look identical.
    """
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", *arguments],
        cwd=RUN_REPO, capture_output=True, text=True,
    )
    if done.returncode != 0:
        print(done.stdout[-4000:])
        print(done.stderr[-4000:])
        raise RuntimeError(
            f"pip install {' '.join(arguments)} failed (exit {done.returncode}); "
            "the output above says why."
        )
    return done


pip_install("-q", "-e", ".[test]")
print("installed from", RUN_REPO)


In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=RUN_REPO,
                        capture_output=True, text=True)
print(result.stdout[-6000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "CPU test suite failed; stop here and read the output above."

In [ ]:
# The profiles, as the code defines them, so the table below is not hand-copied.
from boltz import opt

for name in ("off", "exact", "fast"):
    print(f"{name:6s} -> {' '.join(sorted(opt.PROFILES[name])) or 'none'}")

## 3. Weights and inputs

`BOLTZ_CACHE` holds `boltz2_conf.ckpt`, `boltz2_aff.ckpt`, `ccd.pkl` and `mols/` — about 3 GB.
Point it at Drive to keep it across sessions.

In [ ]:
#@title Weights cache { display-mode: "form" }
CACHE_DIR = "/content/boltz_weights"  #@param {type:"string"}

CACHE = Path(CACHE_DIR)
CACHE.mkdir(parents=True, exist_ok=True)

from boltz.main import download_boltz2
download_boltz2(CACHE)
print(sorted(p.name for p in CACHE.iterdir()))

In [ ]:
INPUTS = Path("/content/inputs"); INPUTS.mkdir(exist_ok=True)

# PDB 1BRS barnase + barstar, single-sequence: 199 tokens, no MSA server needed, so the
# timings below are model time rather than network time.
(INPUTS / "structure").mkdir(exist_ok=True)
(INPUTS / "structure" / "1brs.yaml").write_text(textwrap.dedent("""\
    version: 1
    sequences:
      - protein:
          id: A
          sequence: AQVINTFDGVADYLQTYHKLPDNYITKSEAQALGWVASKGNLADVAPGKSIGGDIFSNREGKLPGKSGRTWREADINYTSGFRNSDRILYSSDWLIYKTTDHYQTFTKIR
          msa: empty
      - protein:
          id: B
          sequence: KKAVINGEQIRSISDLHQTLKKELALPEYYGENLDALWDALTGWVEYPLVLEWRQFEQSKQLTENGAESVLQVFREAKAEGADITIILS
          msa: empty
"""))

# A protein-ligand complex declaring the affinity property: this is what exercises the
# second (affinity) pass, and therefore the structure cache.
(INPUTS / "affinity").mkdir(exist_ok=True)
(INPUTS / "affinity" / "complex.yaml").write_text(textwrap.dedent("""\
    version: 1
    sequences:
      - protein:
          id: A
          sequence: MGSSHHHHHHSSGLVPRGSHMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTRITK
          msa: empty
      - ligand:
          id: B
          smiles: "CC(=O)Oc1ccccc1C(=O)O"
    properties:
      - affinity:
          binder: B
"""))
print(sorted(str(p.relative_to(INPUTS)) for p in INPUTS.rglob("*.yaml")))

## 4. Structure prediction: `off` vs `exact` vs `fast`

Each profile runs in its own subprocess with the same seed and the same settings, so the only
difference between runs is the profile. Wall time includes roughly 20–30 s of start-up,
checkpoint load and featurization that no profile changes — the per-prediction gain is larger
than the end-to-end ratio suggests, and grows with input size and with the number of seeds or
inputs per call.

A first run per profile pays a little extra for CUDA context and kernel selection. The cell
below runs each profile twice and reports the second.

In [ ]:
import re, time

SEED = 42
OUT = Path("/content/out"); OUT.mkdir(exist_ok=True)

def boltz_predict(input_path, out_dir, *, profile, extra=(), env_extra=None):
    """Run one prediction in a clean subprocess and return (seconds, log)."""
    command = [
        sys.executable, "-m", "boltz.main", "predict", str(input_path),
        "--out_dir", str(out_dir), "--cache", str(CACHE),
        "--model", "boltz2", "--seed", str(SEED),
        "--output_format", "mmcif", "--num_workers", "2",
        "--opt_profile", profile, *extra,
    ]
    env = {**os.environ, **(env_extra or {})}
    start = time.perf_counter()
    done = subprocess.run(command, capture_output=True, text=True, env=env)
    seconds = time.perf_counter() - start
    log = done.stdout + done.stderr
    if done.returncode != 0:
        raise RuntimeError(f"profile={profile} failed:\n{log[-4000:]}")
    return seconds, log

if not FORK_GPU_READY:
    print("No sm_80+ GPU: skipping the timed sections. Sections 1-3 already validated the code.")

In [ ]:
structure_timings = {}
structure_logs = {}

if FORK_GPU_READY:
    for profile in ("off", "exact", "fast"):
        out_dir = OUT / f"structure_{profile}"
        for repeat in range(2):
            if out_dir.exists():
                shutil.rmtree(out_dir)
            seconds, log = boltz_predict(
                INPUTS / "structure" / "1brs.yaml", out_dir, profile=profile
            )
        structure_timings[profile] = seconds   # the warm run
        structure_logs[profile] = log
        lever_line = next((l for l in log.splitlines() if l.startswith("[boltz-opt]")), "")
        print(f"{profile:6s} {seconds:7.1f} s   {lever_line}")

### Do the outputs agree?

`exact` must be byte-identical to `off`. `fast` re-associates sums in SDPA, in the fused
projections and in TF32, so it is held to a coordinate tolerance instead — the comparison
prints the aligned all-atom RMSD it actually measured, which is the number to look at.

In [ ]:
def compare(reference, candidate, *flags):
    command = [sys.executable, str(RUN_REPO / "scripts/compare_predictions.py"),
               str(reference), str(candidate), "--reference-seed", str(SEED),
               "--candidate-seed", str(SEED), *flags]
    done = subprocess.run(command, capture_output=True, text=True)
    return done.returncode, json.loads(done.stdout or "{}")

if FORK_GPU_READY:
    rc, report = compare(OUT / "structure_off", OUT / "structure_exact", "--exact")
    print("exact vs off  ->", "IDENTICAL" if rc == 0 else "DIFFERS")
    print("   ", report["summary"])
    if rc != 0:
        print(json.dumps([c for c in report["comparisons"] if not c["passed"]][:3], indent=2))

    rc, report = compare(OUT / "structure_off", OUT / "structure_fast",
                         "--structure-rmsd-atol", "1.0", "--atol", "0.05", "--rtol", "0.05")
    print("fast vs off   ->", "within tolerance" if rc == 0 else "outside tolerance")
    for row in report["comparisons"]:
        for detail in row.get("details", []):
            if detail.get("coordinate_metrics"):
                print(f"    model {row['model']}: aligned all-atom RMSD "
                      f"{detail['aligned_all_atom_rmsd']:.3f} A, "
                      f"max atom {detail['max_aligned_atom_displacement']:.3f} A")

### Which lever is worth what

Re-runs `fast` with one lever removed at a time. The difference against full `fast` is that
lever's contribution on this input — useful because their sizes depend strongly on token count
and on the number of diffusion samples.

In [ ]:
lever_cost = {}

if FORK_GPU_READY:
    baseline = structure_timings["fast"]
    for lever in ("flash_attn", "condproj", "ctorskip", "atom_hoist", "resid",
                  "templ_skip", "dit_hoist"):
        out_dir = OUT / f"ablate_{lever}"
        if out_dir.exists():
            shutil.rmtree(out_dir)
        seconds, _ = boltz_predict(
            INPUTS / "structure" / "1brs.yaml", out_dir,
            profile="fast", extra=("--disable_opt", lever),
        )
        lever_cost[lever] = seconds - baseline
        print(f"without {lever:12s} {seconds:7.1f} s   ({seconds - baseline:+.1f} s vs fast)")

## 5. Affinity: the standard two-pass procedure vs the structure cache

Stock Boltz-2 predicts affinity by re-cropping the complex around the ligand and running the
trunk (5 recycling steps), the diffusion sampler and the confidence head *again* before the
affinity head. `--experimental_structure_cache` replaces all of that with the structure pass's
own trunk state and rank-zero pose.

That is a different inference procedure. The cell below therefore reports two things: how much
time it saves, and how far the affinity value moves. Only the second tells you whether it is
usable for your target.

In [ ]:
affinity = {}

def read_affinity(out_dir):
    results = {}
    for path in (out_dir / "predictions").rglob("affinity_*.json"):
        results[path.parent.name] = json.loads(path.read_text())
    return results

if FORK_GPU_READY:
    for label, extra in (
        ("standard", ()),
        ("structure_cache", ("--experimental_structure_cache", "--override")),
    ):
        out_dir = OUT / f"affinity_{label}"
        if out_dir.exists():
            shutil.rmtree(out_dir)
        seconds, log = boltz_predict(
            INPUTS / "affinity" / "complex.yaml", out_dir, profile="fast", extra=extra
        )
        affinity[label] = {"seconds": seconds, "results": read_affinity(out_dir)}
        print(f"{label:16s} {seconds:7.1f} s")

    for record in affinity["standard"]["results"]:
        left = affinity["standard"]["results"][record]
        right = affinity["structure_cache"]["results"].get(record, {})
        print(f"\n{record}")
        print(f"  mode                        {left.get('affinity_inference_mode')!r:>26} "
              f"{right.get('affinity_inference_mode')!r:>26}")
        for key in ("affinity_pred_value", "affinity_probability_binary"):
            a, b = left.get(key), right.get(key)
            if a is not None and b is not None:
                print(f"  {key:26s} {a:26.4f} {b:26.4f}   delta {b - a:+.4f}")

In [ ]:
# The cache is bound to the pose it was written for: corrupt the digest and the next run must
# refuse it rather than silently reuse a stale state.
if FORK_GPU_READY:
    import numpy as np
    from boltz.data.structure_cache import load_structure_cache

    caches = list((OUT / "affinity_structure_cache" / "predictions").rglob("structure_cache_*.npz"))
    print("caches written:", [p.name for p in caches])
    for cache_path in caches:
        record = cache_path.stem.removeprefix("structure_cache_")
        pose = cache_path.parent / f"pre_affinity_{record}.npz"
        size_mib = cache_path.stat().st_size / 2**20
        try:
            load_structure_cache(
                cache_path, record_id="a-different-record", structure_path=pose,
                token_count=1, atom_count=1,
                token_ids=np.array([0]), atom_ids=np.array([0]),
            )
        except ValueError as error:
            print(f"  {cache_path.name}  {size_mib:.1f} MiB  rejects a foreign record: {error}")
        else:
            raise AssertionError("a cache for another record was accepted")

## 6. Results

In [ ]:
if FORK_GPU_READY:
    import pandas as pd

    rows = [{"run": f"structure / {p}", "seconds": t,
             "speedup vs off": structure_timings["off"] / t}
            for p, t in structure_timings.items()]
    rows += [{"run": f"affinity / {k}", "seconds": v["seconds"],
              "speedup vs off": affinity["standard"]["seconds"] / v["seconds"]}
             for k, v in affinity.items()]
    table = pd.DataFrame(rows).set_index("run").round(2)
    display(table)

    if lever_cost:
        display(pd.Series(lever_cost, name="seconds added when disabled").round(2).sort_values())

In [ ]:
if FORK_GPU_READY:
    import matplotlib.pyplot as plt

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

    profiles = list(structure_timings)
    left.bar(profiles, [structure_timings[p] for p in profiles], color="#6b7fd7")
    left.set_ylabel("wall time (s)")
    left.set_title(f"Structure prediction, 199 tokens, seed {SEED}")
    for index, profile in enumerate(profiles):
        left.text(index, structure_timings[profile],
                  f"{structure_timings[profile]:.0f}s", ha="center", va="bottom")

    labels = list(affinity)
    right.bar(labels, [affinity[k]["seconds"] for k in labels], color="#d78b6b")
    right.set_ylabel("wall time (s)")
    right.set_title("Affinity: two-pass vs structure cache")
    for index, label in enumerate(labels):
        right.text(index, affinity[label]["seconds"],
                   f"{affinity[label]['seconds']:.0f}s", ha="center", va="bottom")

    for axis in (left, right):
        axis.spines[["top", "right"]].set_visible(False)
    plt.show()

## What the numbers do and do not say

* Wall time here includes start-up, checkpoint load and featurization, which no profile
  changes. On one 199-token input that overhead dominates; the gains grow with token count,
  with `--diffusion_samples`, and with several inputs or seeds per call.
* `exact` being byte-identical to `off` is checked here on real output files, and in the CPU
  suite on a full diffusion roll-out. That is the claim this fork makes about it.
* `fast` is *not* claimed to be identical. Judge it by the RMSD printed in section 4 against
  what a different seed does to the same input.
* The structure-cache affinity path is an alternative procedure. Section 5's delta on one
  ligand is a smoke test, not a validation. For a real decision, run it across a ligand series
  — `benchmarks/fep4/` has 87 protein-ligand inputs across four targets — and compare ranking,
  not single values.
* `--num_workers`, `--diffusion_samples` and GPU class all move these timings. Keep them fixed
  when comparing.